In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SILVER = ROOT / "datalake_silver"
QUARANTINE = SILVER / "_quarantine"
FIGURES = ROOT / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# 1. Cargar todas las particiones Silver y Cuarentena
fire_files = sorted(SILVER.glob("fire_detections_*.parquet"))
weather_files = sorted(SILVER.glob("weather_hourly_*.parquet"))
rej_files = sorted(QUARANTINE.glob("rejected_*.parquet"))

df_fires = pd.concat([pd.read_parquet(f) for f in fire_files], ignore_index=True)
df_weather = pd.concat([pd.read_parquet(f) for f in weather_files], ignore_index=True)
df_rej = pd.concat([pd.read_parquet(f) for f in rej_files], ignore_index=True) if rej_files else pd.DataFrame()

print(f"Fire partitions: {len(fire_files)} | Total valid rows: {len(df_fires):,}")
print(f"Weather partitions: {len(weather_files)} | Total valid rows: {len(df_weather):,}")
print(f"Quarantine partitions: {len(rej_files)} | Total rejected rows: {len(df_rej):,}")

# 2. Motivos de cuarentena
if not df_rej.empty:
    print("\n--- Quarantine Reasons ---")
    print(df_rej["reason"].value_counts(dropna=False))
    print((df_rej["reason"].value_counts(normalize=True) * 100).round(2))

# 3. Tasas de nulos por columna
print("\n--- Fire Null Rates (%) ---")
print((df_fires.isna().mean() * 100).round(2)[lambda s: s > 0])

print("\n--- Weather Null Rates (%) ---")
print((df_weather.isna().mean() * 100).round(2)[lambda s: s > 0])

# 4. Estadísticas descriptivas y outliers en FRP / variables numéricas
print("\n--- Fire Numeric Stats ---")
for col in ["frp", "bright_ti4", "bright_ti5"]:
    s = df_fires[col].dropna()
    q25, q50, q75 = np.percentile(s, [25, 50, 75])
    out_pct = (df_fires["frp_outlier"].mean() * 100) if col == "frp" and "frp_outlier" in df_fires else 0.0
    print(f"{col}: count={len(s)}, null%={df_fires[col].isna().mean()*100:.2f}, mean={s.mean():.2f}, median={q50:.2f}, IQR={q75-q25:.2f}, outlier%={out_pct:.2f}")

# 5. Revisión de status (new vs revised), flags e imputación en clima
print("\n--- Fire Record Status ---")
print(df_fires["record_status"].value_counts(dropna=False))
print("\n--- Fire Quality Flags ---")
print(df_fires["quality_flags"].value_counts(dropna=False).head(10))

if "imputed" in df_weather.columns:
    print("\n--- Weather Imputed Rate (%) ---")
    print((df_weather.groupby("variable", observed=True)["imputed"].mean() * 100).round(2))
    print("\n--- Weather Numeric Stats ---")
    for var_name, grp in df_weather.groupby("variable", observed=True):
        s = grp["value"].dropna()
        q25, q50, q75 = np.percentile(s, [25, 50, 75])
        z_pct = grp["quality_flags"].str.contains("zscore_gt4", na=False).mean() * 100
        print(f"{var_name}: count={len(s)}, null%={grp['value'].isna().mean()*100:.2f}, mean={s.mean():.2f}, median={q50:.2f}, IQR={q75-q25:.2f}, outlier%={z_pct:.2f}")

# 6. Generar figura diagnóstica de 4 paneles para el reporte LaTeX
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if not df_rej.empty:
    df_rej["reason"].value_counts().plot(kind="bar", ax=axes[0, 0], color="#c0392b")
    axes[0, 0].set_title("(a) Quarantine Rejection Reasons")
    axes[0, 0].set_ylabel("Records")
    axes[0, 0].tick_params(axis="x", rotation=20)

for sat, grp in df_fires.groupby("satellite", observed=True):
    np.log10(grp["frp"].clip(lower=0.01)).plot(kind="hist", bins=30, alpha=0.55, label=str(sat), ax=axes[0, 1])
axes[0, 1].set_title("(b) log10(FRP) Distribution by Satellite")
axes[0, 1].set_xlabel("log10(FRP [MW])")
axes[0, 1].legend()

ct = pd.crosstab(df_fires["confidence_level"], df_fires["daynight"])
ct.plot(kind="bar", ax=axes[1, 0], color=["#f39c12", "#2c3e50"])
axes[1, 0].set_title("(c) Detections by Confidence Level and Day/Night")
axes[1, 0].set_ylabel("Detections")
axes[1, 0].tick_params(axis="x", rotation=0)

df_fires["region_name"].value_counts().head(10).sort_values().plot(kind="barh", ax=axes[1, 1], color="#27ae60")
axes[1, 1].set_title("(d) Top 10 Departments by Valid Silver Detections")
axes[1, 1].set_xlabel("Detections")

plt.tight_layout()
plt.savefig(FIGURES / "silver_dq_overview.png", dpi=150)
print(f"\nSaved figure to: {FIGURES / 'silver_dq_overview.png'}")

Fire partitions: 4 | Total valid rows: 976
Weather partitions: 4 | Total valid rows: 60,192
Quarantine partitions: 4 | Total rejected rows: 5,310

--- Quarantine Reasons ---
reason
outside_colombia    5310
Name: count, dtype: int64
reason
outside_colombia    100.0
Name: proportion, dtype: float64

--- Fire Null Rates (%) ---
Series([], dtype: float64)

--- Weather Null Rates (%) ---
Series([], dtype: float64)

--- Fire Numeric Stats ---
frp: count=976, null%=0.00, mean=7.31, median=5.68, IQR=4.90, outlier%=1.33
bright_ti4: count=976, null%=0.00, mean=335.14, median=335.61, IQR=10.47, outlier%=0.00
bright_ti5: count=976, null%=0.00, mean=294.44, median=295.90, IQR=6.94, outlier%=0.00

--- Fire Record Status ---
record_status
new    976
Name: count, dtype: int64[pyarrow]

--- Fire Quality Flags ---
quality_flags
                   963
frp_outlier_iqr     13
Name: count, dtype: int64[pyarrow]

--- Weather Imputed Rate (%) ---
variable
precipitation           0.0
relative_humidity_2m    0.


Saved figure to: C:\Proyectos\wildfire-monitoring\docs\figures\silver_dq_overview.png
